# 04 Load Schema v0.1 into Neo4j Aura (v2: source-record preserving)

**Date:** 2026-10-15, revised 2026-10-22
**Author:** Xincheng (George) Ding. v2 applies fixes from Deniz Özçakır's review (denizbranch 3c1d023): every node and relationship carries a `source_id` (`<table>.csv:row:<n>`) and relationships are merged on that id, so the 807 participation rows that v1 collapsed (same organisation and activity, different role) are preserved.
**Needs network** (Aura Free). Fill `NEO4J_*` in `fa26/.env` and set `IATI_ALLOW_DB_LOAD=1`.

**Purpose:** Load the pinned benchmark snapshot (`fa26/benchmark/snapshot/`, hash-verified) into Aura. Expected result: 27,279 nodes, 59,121 relationships, 0 isolated nodes. The `source_id` values are the citation IDs the 20-question benchmark scores against, so notebook 05 depends on this load.

**Re-loading:** v1 relationships have no `source_id`, so merging v2 on top of a v1 load would duplicate every edge. Set `IATI_WIPE_DB=1` to clear the database first (batched DETACH DELETE). Aura Free allows one instance, so this notebook wipes rather than creating an isolated database.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fa26" / "benchmark" / "manifest.json").exists())
FA26 = ROOT / "fa26"
sys.path.insert(0, str(FA26 / "benchmark")); sys.path.insert(0, str(FA26 / "src"))
from common import verify_snapshot
SNAPSHOT = verify_snapshot(); print("snapshot", SNAPSHOT["snapshot_id"][:16], "verified")
GRAPH_DIR = FA26 / "benchmark" / "snapshot"

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase
load_dotenv(FA26 / ".env", override=True)
if os.environ.get("IATI_ALLOW_DB_LOAD") != "1":
    raise RuntimeError("Set IATI_ALLOW_DB_LOAD=1 in fa26/.env to confirm this database may be (re)loaded")
driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))
driver.verify_connectivity(); print("Neo4j connected")
DB = os.environ.get("NEO4J_DATABASE", "neo4j")

tables = {p.stem: pd.read_csv(p, dtype=str, keep_default_na=False) for p in sorted(GRAPH_DIR.glob("*.csv"))}
for name, frame in tables.items():
    frame["source_id"] = [f"{name}.csv:row:{i}" for i in range(1, len(frame) + 1)]
print({k: len(v) for k, v in tables.items() if len(v)})

In [ ]:
# Optional wipe. Required when the database still holds a v1 load (relationships without source_id).
if os.environ.get("IATI_WIPE_DB") == "1":
    with driver.session(database=DB) as s:
        before = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
        while True:
            deleted = s.run("MATCH (n) WITH n LIMIT 5000 DETACH DELETE n RETURN count(*) AS d").single()["d"]
            if deleted == 0: break
        print(f"wiped {before} nodes")
else:
    with driver.session(database=DB) as s:
        n = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
        bad = s.run("MATCH ()-[r]->() WHERE r.source_id IS NULL RETURN count(r) AS c").single()["c"]
    print(f"existing nodes: {n}; relationships without source_id: {bad}")
    assert bad == 0, "Database holds v1 relationships. Set IATI_WIPE_DB=1 and rerun this cell."

In [ ]:
from iati_codelists import TRANSACTION_TYPE, ORG_TYPE, SECTOR_VOCAB

def prep(csv):
    df = tables[csv].copy()
    if csv == "nodes_transactions":
        df["type_label"] = df["type"].map(TRANSACTION_TYPE).fillna("Unknown")
        df["value_num"] = pd.to_numeric(df["value"], errors="coerce")
    if csv == "nodes_organisations": df["type_label"] = df["type"].map(ORG_TYPE).fillna("Unknown")
    if csv == "nodes_sectors":       df["vocabulary_label"] = df["vocabulary"].map(SECTOR_VOCAB).fillna("Unknown")
    if csv == "nodes_activities":    df["hunger_basis"] = df["match_basis"]
    return df.astype(object).where(df.notna(), None)

NODES = {"nodes_activities": ("Activity", "iati_identifier"), "nodes_organisations": ("Organisation", "ref"),
         "nodes_countries": ("Country", "code"), "nodes_sectors": ("Sector", "sector_key"),
         "nodes_locations": ("Location", "location_id"), "nodes_transactions": ("Transaction", "tx_id"),
         "nodes_budgets": ("Budget", "budget_id"), "nodes_sdgs": ("SDG", "sdg_key")}
RELS = [("rel_reports", "REPORTS", "Organisation", "org_ref", "ref", "Activity", "iati_identifier", "iati_identifier", None),
        ("rel_participates", "PARTICIPATES_IN", "Organisation", "org_ref", "ref", "Activity", "iati_identifier", "iati_identifier", None),
        ("rel_targets_country", "TARGETS", "Activity", "iati_identifier", "iati_identifier", "Country", "country_code", "code", None),
        ("rel_classified_as", "CLASSIFIED_AS", "Activity", "iati_identifier", "iati_identifier", "Sector", "sector_key", "sector_key", None),
        ("rel_located_at", "LOCATED_AT", "Activity", "iati_identifier", "iati_identifier", "Location", "location_id", "location_id", None),
        ("rel_contributes_to", "CONTRIBUTES_TO", "Activity", "iati_identifier", "iati_identifier", "SDG", "sdg_key", "sdg_key", None),
        ("nodes_transactions", "HAS_TRANSACTION", "Activity", "iati_identifier", "iati_identifier", "Transaction", "tx_id", "tx_id", None),
        ("nodes_budgets", "HAS_BUDGET", "Activity", "iati_identifier", "iati_identifier", "Budget", "budget_id", "budget_id", None),
        ("rel_transaction_edges", "PROVIDED_BY", "Transaction", "tx_id", "tx_id", "Organisation", "org_ref", "ref", ("edge", "PROVIDED_BY")),
        ("rel_transaction_edges", "RECEIVED_BY", "Transaction", "tx_id", "tx_id", "Organisation", "org_ref", "ref", ("edge", "RECEIVED_BY"))]

In [ ]:
DDL = [
 "CREATE CONSTRAINT activity_id  IF NOT EXISTS FOR (n:Activity)     REQUIRE n.iati_identifier IS UNIQUE",
 "CREATE CONSTRAINT org_ref      IF NOT EXISTS FOR (n:Organisation) REQUIRE n.ref IS UNIQUE",
 "CREATE CONSTRAINT country_code IF NOT EXISTS FOR (n:Country)      REQUIRE n.code IS UNIQUE",
 "CREATE CONSTRAINT sector_key   IF NOT EXISTS FOR (n:Sector)       REQUIRE n.sector_key IS UNIQUE",
 "CREATE CONSTRAINT location_id  IF NOT EXISTS FOR (n:Location)     REQUIRE n.location_id IS UNIQUE",
 "CREATE CONSTRAINT tx_id        IF NOT EXISTS FOR (n:Transaction)  REQUIRE n.tx_id IS UNIQUE",
 "CREATE CONSTRAINT budget_id    IF NOT EXISTS FOR (n:Budget)       REQUIRE n.budget_id IS UNIQUE",
 "CREATE CONSTRAINT sdg_key      IF NOT EXISTS FOR (n:SDG)          REQUIRE n.sdg_key IS UNIQUE",
 "CREATE INDEX activity_publisher IF NOT EXISTS FOR (n:Activity)     ON (n.publisher)",
 "CREATE INDEX activity_hunger    IF NOT EXISTS FOR (n:Activity)     ON (n.hunger_basis)",
 "CREATE INDEX org_name           IF NOT EXISTS FOR (n:Organisation) ON (n.name)",
 "CREATE INDEX tx_type            IF NOT EXISTS FOR (n:Transaction)  ON (n.type)",
 "CREATE INDEX tx_date            IF NOT EXISTS FOR (n:Transaction)  ON (n.date)",
 "CREATE INDEX country_tier       IF NOT EXISTS FOR (n:Country)      ON (n.hotspot_tier)",
] + [f"CREATE INDEX rel_{r[1].lower()}_source IF NOT EXISTS FOR ()-[x:{r[1]}]-() ON (x.source_id)" for r in RELS]
with driver.session(database=DB) as s:
    for q in DDL: s.run(q)
print(f"{len(DDL)} constraints and indexes ready")

In [ ]:
def batched(recs, size=2000):
    for i in range(0, len(recs), size): yield recs[i:i + size]

with driver.session(database=DB) as s:
    for csv, (label, key) in NODES.items():
        recs = prep(csv).to_dict("records")
        for b in batched(recs):
            s.run(f"UNWIND $rows AS r MERGE (n:{label} {{{key}: r.{key}}}) SET n += r", rows=b)
        print(f"{label:<14} {len(recs):>7} nodes")
    for csv, rtype, fl, fc, fk, tl, tc, tk, flt in RELS:
        df = tables[csv]
        if flt: df = df[df[flt[0]] == flt[1]]
        props = [c for c in df.columns if c not in (fc, tc, "edge", "source_id")] if csv.startswith("rel_") else []
        rows = [{"f": r[fc], "t": r[tc], "sid": r["source_id"], "p": {k: r[k] for k in props}} for r in df.to_dict("records")]
        q = (f"UNWIND $rows AS r MATCH (a:{fl} {{{fk}: r.f}}) MATCH (b:{tl} {{{tk}: r.t}}) "
             f"MERGE (a)-[x:{rtype} {{source_id: r.sid}}]->(b) SET x += r.p")
        for b in batched(rows): s.run(q, rows=b)
        print(f"{rtype:<16} {len(rows):>7} relationships")

In [ ]:
EXPECTED_NODES = sum(len(tables[c]) for c in NODES)
EXPECTED_RELS = sum(len(tables[c]) if not flt else int((tables[c][flt[0]] == flt[1]).sum()) for c, *_, flt in RELS)
with driver.session(database=DB) as s:
    nodes = pd.DataFrame(s.run("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS n ORDER BY n DESC").data())
    rels = pd.DataFrame(s.run("MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS n ORDER BY n DESC").data())
    isolated = s.run("MATCH (n) WHERE NOT (n)--() RETURN count(n) AS c").single()["c"]
    roles = s.run("MATCH (o:Organisation)-[p:PARTICIPATES_IN]->(a:Activity) WITH o, a, count(p) AS k WHERE k > 1 RETURN count(*) AS pairs").single()["pairs"]
print(nodes.to_string(index=False)); print(rels.to_string(index=False))
print(f"nodes {nodes.n.sum()} (expected {EXPECTED_NODES}) | relationships {rels.n.sum()} (expected {EXPECTED_RELS}) | isolated {isolated} | org/activity pairs with >1 role {roles}")
assert nodes.n.sum() == EXPECTED_NODES and rels.n.sum() == EXPECTED_RELS and isolated == 0
driver.close()